# sleep: records, coverage and defensible relationships
For Friday October 23, 2026. Only this table and the small/aggregated comparisons needed below are read, inside a read-only repeatable-read snapshot. No database changes are made. Outputs contain aggregate statistics and anonymous plots, not participant identifiers or raw records.

Shared IDs/group/type identify context; calendar/week/day label the export/reporting period; start/end/duration describe its window. Score definitions, label rules, timestamp conventions and device algorithms are unverified without the source dictionary. Names and SQL types are not definitions. No clinical thresholds, diagnosis, causation or predictive validation are used.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display, JSON, Markdown
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately in .env.'
engine=create_engine(url); del url
connection=engine.connect().execution_options(isolation_level='REPEATABLE READ')
transaction=connection.begin()
connection.execute(text('SET TRANSACTION READ ONLY'))
connection.execute(text("SET LOCAL statement_timeout = '240s'"))
def query(sql): return pd.read_sql(text(sql),connection)
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
results={}
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
def show_schema(table):
    cols=inspect(connection).get_columns(table)
    display(pd.DataFrame([{'column':c['name'],'SQL_type':str(c['type'])} for c in cols]))
    return [c['name'] for c in cols]
def audit_keys(df,candidates):
    display(pd.DataFrame([{'key':name,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
      'rows_in_duplicate_keys':int(df.duplicated(k,keep=False).sum()),'unique_nonnull':valid_key(df,k)} for name,k in candidates.items()]))
def period_audit(p):
    p=p.copy(); p['calendar_date']=pd.to_datetime(p.calendar_date,errors='coerce')
    elapsed=p.end_measure_period_sec-p.start_measure_period_sec
    ordered=p.sort_values(key)
    prior=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
    utc=pd.to_datetime(p.start_measure_period_sec,unit='s',utc=True,errors='coerce')
    stats={'periods':len(p),'participants':p.sdv_user_id.nunique(),
      'null_key_rows':int(p[key].isna().any(axis=1).sum()),'duplicate_period_keys':int(p.duplicated(key,keep=False).sum()),
      'nonpositive_periods':int(elapsed.le(0).sum()),'exactly_24_hour_periods':int(elapsed.eq(86400).sum()),
      'overlapping_periods':int(ordered.start_measure_period_sec.lt(prior).sum()),
      'UTC_start_date_matches_label':int(utc.dt.tz_localize(None).dt.normalize().eq(p.calendar_date).sum())}
    if 'duration_measure_period_sec' in p:
        stats['duration_equals_end_minus_start']=int(p.duration_measure_period_sec.eq(elapsed).sum())
    display(pd.DataFrame([stats])); display((elapsed/3600).value_counts().sort_index().rename_axis('period_hours').to_frame('periods'))
    coverage=p.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),observed_dates=('calendar_date','nunique'),periods=('calendar_date','size'))
    coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
    coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
    coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
    display(coverage[['observed_dates','periods','span_days','missing_dates_within_span','observed_date_percent']].describe().T)
    stats.update({'min_observed_dates_per_participant':int(coverage.observed_dates.min()),
      'max_observed_dates_per_participant':int(coverage.observed_dates.max()),
      'min_observed_date_percent':coverage.observed_date_percent.min(),'max_observed_date_percent':coverage.observed_date_percent.max()})
    return stats
def checked_join(left,right,label):
    if not (valid_key(left,key) and valid_key(right,key)):
        print(label+': skipped; keys are not unique and non-null.'); return None
    joined=left.merge(right,on=key,how='outer',suffixes=('_left','_right'),indicator=True,validate='one_to_one')
    counts={str(k):int(v) for k,v in joined._merge.value_counts().items()}
    results[label+'_period_join']=counts
    results[label+'_participant_overlap']={'left':left.sdv_user_id.nunique(),'right':right.sdv_user_id.nunique(),
      'shared':len(set(left.sdv_user_id.dropna()) & set(right.sdv_user_id.dropna()))}
    display(pd.Series(counts,name='periods').rename_axis(label+'_status').to_frame())
    return joined

TABLE='sleep'
columns=show_schema(TABLE)
d=query('SELECT * FROM '+TABLE)
d['calendar_date']=pd.to_datetime(d.calendar_date,errors='coerce')
results['inventory']={'rows':len(d),'participants':d.sdv_user_id.nunique(),'null_cells':int(d.isna().sum().sum()),'exact_duplicate_rows_after_first':int(d.duplicated().sum())}
display(pd.DataFrame([results['inventory']]))
display(pd.DataFrame({'column':d.columns,'pandas_type':d.dtypes.astype(str).values,'missing_rows':d.isna().sum().values,'distinct_nonnull':d.nunique().values}))


## 1. Summary grain, keys and date coverage
Test record ID and participant/date/period keys before treating a row as one summary. A date label does not prove a complete night or full wear. Missing summaries remain absent, not zero. Inspect exact duplicates and nulls separately from operational sentinel/label checks.

In [ ]:
audit_keys(d,{'record_id':['additional_id'],'participant_date':['sdv_user_id','calendar_date'],'period':key})
results['periods']=period_audit(d)


## Reporting-window interpretation
Reporting windows are checked separately from actual observations. End minus start, nonpositive windows, exact 24-hour windows and overlaps are diagnostics, not proof of wear or complete days. UTC conversion assumes Unix seconds only as a plausibility check; local calendar labels and inclusive/exclusive endpoints need documentation. Date coverage is unique observed dates divided by dates in each participant's first-to-last observed span; enrolment boundaries are unknown. Missing dates are never zero values. Participant identifiers are retained only in memory for grouping and are not displayed.

## 2. Establish whether fields are labels or numbers
Despite their names, `total_duration`, `stress`, `awake`, `remperc`, `restless`, `lightperc` and `deepperc` must be inspected before arithmetic. Count distinct recorded labels and test numeric parseability without replacing the original strings. If these are labels such as FAIR or INVALID, they are neither seconds nor percentages and must not be summed/averaged as such. INVALID is retained as an explicit category, not silently converted to missing or zero. No ordinal encoding is invented.

In [ ]:
label_fields=['total_duration','stress','awake','remperc','restless','lightperc','deepperc']
checks=[]
for field in label_fields:
    numeric=pd.to_numeric(d[field],errors='coerce')
    checks.append({'field':field,'missing':int(d[field].isna().sum()),'numeric_parseable':int(numeric.notna().sum()),
      'distinct_nonnull':d[field].nunique(),'INVALID':int(d[field].eq('INVALID').sum())})
    display(d[field].fillna('<missing>').value_counts().rename_axis(field).to_frame('records'))
display(pd.DataFrame(checks)); results['label_checks']=checks
results['rows_with_any_INVALID_label']=int(d[label_fields].eq('INVALID').any(axis=1).sum())
results['all_fields_numeric_parseable']=bool(pd.DataFrame({f:pd.to_numeric(d[f],errors='coerce') for f in label_fields}).notna().all(axis=None))
counts=pd.DataFrame({field:d[field].fillna('<missing>').value_counts() for field in label_fields}).fillna(0)
fig,ax=plt.subplots(figsize=(11,4)); counts.T.plot.bar(stacked=True,ax=ax)
ax.set(title='Recorded label frequencies (not numeric durations/percentages)',xlabel='Source field',ylabel='Records'); ax.tick_params(axis='x',rotation=25)
ax.legend(title='Recorded label',bbox_to_anchor=(1.02,1),loc='upper left')
plt.tight_layout(); plt.show()
display(pd.crosstab(d.remperc,d.deepperc))


## 3. Validated relationships: summary pairing and event coverage
Compare the two summary tables on participant/date/start/end with unique/non-null keys and explicit one-to-one cardinality. Check shared ID/group/duration. For events, aggregate to one row per exact period first, retaining context-variation counts; never merge every event directly with repeated summaries.

The event table may cover more participants and periods. Unmatched counts quantify coverage differences rather than zero scores or no events. Calendar dates may label a reporting day, not the night-start date. This cell counts event rows/periods only; the dedicated event notebook handles interval unions, cross-midnight and overlap.

In [ ]:
other=query('SELECT * FROM overall_sleep'); other['calendar_date']=pd.to_datetime(other.calendar_date,errors='coerce')
audit_keys(other,{'period':key,'participant_date':['sdv_user_id','calendar_date']})
j=checked_join(d,other,'overall_sleep')
if j is not None:
    both=j._merge.eq('both')
    results['summary_context']={field+'_equal':int((both & j[field+'_left'].eq(j[field+'_right'])).sum()) for field in ['additional_id','sdv_group_id','duration_measure_period_sec']}
    display(pd.crosstab(j.loc[both,'total_duration'],j.loc[both,'sleep_qualifier']))
    display(j.loc[both].groupby('total_duration').sleep_value.agg(['count','median','min','max']))
events=query("""SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,
 COUNT(*) event_rows,COUNT(DISTINCT additional_id) record_variants,MIN(additional_id) additional_id,
 COUNT(DISTINCT sdv_group_id) group_variants,MIN(sdv_group_id) sdv_group_id,
 COUNT(DISTINCT duration_measure_period_sec) duration_variants,MIN(duration_measure_period_sec) duration_measure_period_sec
 FROM sleep_level_events GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec""")
events['calendar_date']=pd.to_datetime(events.calendar_date,errors='coerce')
ej=checked_join(d,events,'sleep_level_events')
if ej is not None:
    both=ej._merge.eq('both'); unique=both & ej.record_variants.eq(1) & ej.group_variants.eq(1) & ej.duration_variants.eq(1)
    results['event_context']={field+'_equal':int((unique & ej[field+'_left'].eq(ej[field+'_right'])).sum()) for field in ['additional_id','sdv_group_id','duration_measure_period_sec']}
results['event_coverage']={'events':int(events.event_rows.sum()),'periods':len(events),'participants':events.sdv_user_id.nunique()}


## 4. Descriptive label pairing with the overall score
Use the validated matched summary records only. Compare the distribution of the recorded overall score across `total_duration` labels without recoding labels into durations or claiming that the label causes the score. Records repeat within participants, follow-up differs and score/labels may share an underlying algorithm. Other component fields likewise remain categories.

In [ ]:
if j is not None:
    matched=j.loc[j._merge.eq('both')]
    groups=[(label,g.sleep_value.dropna()) for label,g in matched.groupby('total_duration')]
    fig,ax=plt.subplots(figsize=(8,3.5)); ax.boxplot([v for _,v in groups],tick_labels=[str(k) for k,_ in groups])
    ax.set(title='Overall recorded score by total_duration label',xlabel='Recorded category (not hours)',ylabel='Overall score; scale unverified')
    plt.tight_layout(); plt.show()


## Verified findings and teacher takeaway

- **299 records/five participants**, with no null cells or exact duplicates. Record ID, participant/date and exact period keys are unique/non-null.
- `total_duration`, `stress`, `awake`, `remperc`, `restless`, `lightperc` and `deepperc` are **categorical text labels** in this snapshot: zero numeric-parseable values in every field. They must not be treated as hours, seconds or numeric percentages despite their names.
- The categories include EXCELLENT, GOOD, FAIR and POOR. **Two records contain INVALID** in each of `remperc`, `lightperc` and `deepperc`; the same two records account for all INVALID labels. INVALID is retained rather than silently recoded or dropped. No ordinal encoding or score threshold is inferred.
- **293 reporting periods are exactly 24 hours**, six are not; all durations equal end minus start, with no reporting-window overlap. Date coverage within participant observed spans ranges 78.57%–99.18%, not enrolment/wear coverage.
- All **299 records match overall_sleep** one-to-one on participant/date/start/end, with agreeing IDs/groups/durations and the same five participants.
- All **299 summary periods match aggregated sleep-level events**; **498 event periods have no sleep-summary match**. Event coverage is 797 periods/eight participants. This is unequal export/cohort coverage, not zero sleep or an estimated missing percentage.

**Takeaway:** “The sleep component table contains ratings, not numeric sleep durations or stage percentages. I verified label types and INVALID records first, then matched it to the overall score and event coverage using exact period keys.”

Source rating rules, score generation, night/date assignment, timezone and wear coverage remain unverified. Repeated records and shared source algorithms limit interpretation of label/score comparisons.


## Teacher-facing method and unresolved meanings
“I established whether rows were summaries or events, checked missing/duplicate/invalidity candidates, and validated period keys before any comparison. I kept unequal participant coverage and unmatched records visible.”

For events, sums can double-count overlapping intervals; interval unions use an explicit diagnostic convention. For sleep strings, labels are not numeric durations or percentages. Source score/impact/label definitions, algorithms, endpoints, timezone, enrolment and actual wear remain unresolved. Repeated records and unequal follow-up limit pooled interpretation. No medical, causal or predictive conclusion is made. CSV export, RAG and model building remain deferred.

In [ ]:
# End the read-only snapshot. No database changes were made.
transaction.rollback(); connection.close(); engine.dispose()
def scalar(v):
    if hasattr(v,'item'): return v.item()
    return str(v)
safe_results=json.loads(json.dumps(results,default=scalar))
display(JSON(safe_results))
print('All analysis cells completed; read-only connection closed.')
